# SpeechLM API in vLLM example

In [ ]:
# example of llm streaming usage

from operator import ne
from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM

import torch

model_path = "/home/vklimkov/.cache/huggingface/hub/models--TinyLlama--TinyLlama-1.1B-Chat-v1.0/snapshots/fe8a4ea1ffedaf415f4da2f062534de366a451e6/"
engine_args = AsyncEngineArgs(
    model=model_path,
    max_model_len=256,
    gpu_memory_utilization=0.8,
    #enforce_eager=True,
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=20, skip_sampling=True)

tokens = engine.tokenizer.encode("My name is")
inputs = {
    "prompt_token_ids": [0] * len(tokens),
    "custom_inputs": {
        "custom_in_tokens": torch.tensor(tokens, dtype=torch.int32),
    }
}

sampled_tokens = []
async for output in engine.generate(
    inputs,
    sampling_params=sampling_params,
    request_id="1",
):
    new_token = output.outputs[0].custom_outputs["custom_out_tokens"][-1].item()
    sampled_tokens.append(new_token)
    if not output.finished:
        await engine.append_request(
            request_id="1",
            custom_inputs={"custom_in_tokens": torch.tensor([new_token], dtype=torch.int32)}
        )

print(f">>> sampledtokens: [{str(sampled_tokens)}]", flush=True)
text = engine.tokenizer.decode(sampled_tokens)
print(f">>> sampledtext: [{text}]", flush=True)

In [ ]:
# Same example, but with shared-memory decode channel enabled.
#
# When shm_decode=True, add_request() automatically creates a SHM
# channel.  Each decode iteration uses a single decode_step() call
# that writes inputs to shared memory, signals the core, and waits
# for the output — all in one await.
#
# Requirements:
#   - Model config.json must define custom_input_specs AND
#     custom_output_specs.

from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM
import torch

engine_args = AsyncEngineArgs(
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_model_len=256,
    gpu_memory_utilization=0.8,
    shm_decode=True,  # enable shared-memory decode channel
    input_coalesce_timeout_ms=5,
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=100, skip_sampling=True)

tokens = engine.tokenizer.encode("My name is")
inputs = {
    "prompt_token_ids": [0] * len(tokens),
    "custom_inputs": {
        "custom_in_tokens": torch.tensor(tokens, dtype=torch.int32),
    },
}

request_id = "shm_1"

# 1. Submit the request.  shm_decode=True means add_request()
#    internally creates and registers a SHM channel.
queue = await engine.add_request(request_id, inputs, sampling_params)

# 2. Prefill output arrives via the normal ZMQ/output-processor path.
prefill_output = await queue.get()
last_token = prefill_output.outputs[0].custom_outputs["custom_out_tokens"][-1].item()


sampled_tokens = [last_token]

# 3. Decode loop — single decode_step() call per iteration.
#    Writes inputs to SHM, signals the core, blocks in futex until
#    output is ready, and returns.  Synchronous — no asyncio bounce.
for _ in range(sampling_params.max_tokens - 1):
    outputs = engine.decode_step_shm(
        request_id,
        custom_inputs={
            "custom_in_tokens": torch.tensor([last_token], dtype=torch.int32),
        },
    )
    last_token = int(outputs["custom_out_tokens"][0])
    sampled_tokens.append(last_token)

# 4. Clean up (also closes the SHM channel).
await engine.abort(request_id)


print(f">>> sampledtokens: [{str(sampled_tokens)}]", flush=True)
text = engine.tokenizer.decode(sampled_tokens)
print(f">>> sampledtext: [{text}]", flush=True)